# 적분 개념 재발견 실험 — 4·5단계 (강화 학습 싼 버전)
**이 노트북이 하는 일:** 저장소를 내려받고 → 패키지 설치 → 3단계에서 만든 가중치를 불러와 → 2단계 학습(잘 된 풀이만 모아서 따라하기)을 돌립니다.

**처음 한 번만 할 일**
1. 런타임 → 런타임 유형 변경 → GPU(T4) 선택.
2. 저장소가 비공개라면: 왼쪽 열쇠 아이콘(보안 비밀/Secrets)에 `GITHUB_TOKEN` 이름으로 토큰을 넣고 "노트북 액세스"를 켭니다. (토큰을 노트북이나 저장소에 직접 적지 마세요.)
3. 아래 셀을 위에서부터 차례로 실행.

**끊겼을 때:** 12시간 안에 끊길 수 있습니다. 다시 열어서 *같은 셀들을 처음부터 다시 실행*하면, 드라이브에 저장된 마지막 지점(50바퀴마다 저장)부터 이어서 돕니다.

## 1. 구글 드라이브 연결 (결과가 여기에 저장됩니다)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/integral_runs/run1'   # 실험을 새로 시작하려면 이 이름을 바꾸세요
import os; os.makedirs(RUN_DIR, exist_ok=True)
print(RUN_DIR)

## 2. 저장소 내려받기

In [ ]:
import os, subprocess
REPO = 'gitdd12/integral_discovery'
BRANCH = 'ccr-206f948d-nkcq34'      # 바꾸고 싶으면 여기 (예: main)
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    url = f'https://{token}@github.com/{REPO}.git'
except Exception:
    url = f'https://github.com/{REPO}.git'      # 공개 저장소면 토큰 없이도 됨
if not os.path.exists('/content/integral_discovery'):
    subprocess.run(['git','clone','-b',BRANCH,url,'/content/integral_discovery'], check=True)
else:
    subprocess.run(['git','-C','/content/integral_discovery','pull'], check=True)
%cd /content/integral_discovery
!git log --oneline | head -3

## 3. 패키지 확인 (코랩에는 대부분 이미 있음)

In [ ]:
!pip -q install numpy scipy
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())

## 4. 테스트 (모두 통과해야 다음으로)

In [ ]:
!pip -q install pytest
!PYTHONPATH=. python -m pytest tests -q

## 5. 시범 운행 (200바퀴)
점수가 전혀 안 오르면 멈추고 알려주세요. 한 바퀴에 몇 초 걸리는지 보고 본 학습 시간을 다시 추정합니다.

In [ ]:
!PYTHONPATH=. python scripts/train_rl.py --init weights/school.pt --out {RUN_DIR} --rounds 200 --thought_first 20

## 6. 본 학습
시범 운행 결과를 같이 본 뒤에 돌립니다. `--rounds` 를 늘려서 같은 폴더(RUN_DIR)로 다시 실행하면 이어서 갑니다.

In [ ]:
# !PYTHONPATH=. python scripts/train_rl.py --init weights/school.pt --out {RUN_DIR} --rounds 20000 --thought_every 1000

## 7. 점수 보기

In [ ]:
import json, numpy as np
rows = [json.loads(l) for l in open(f'{RUN_DIR}/scores.jsonl')]
print(len(rows), '바퀴')
for r in rows[::max(1, len(rows)//15)]:
    print(f"바퀴 {r['round']:5d}  평균점수 {r['mean']:.3f}  최고 {r['max']:.2f}  유효 {r['valid']*100:4.0f}%  제출 {r['submits']:.1f}회  반복사용 {r['loops']*100:3.0f}%  {r['secs']}s")

## 8. 결과를 Claude Code에 넘기기
`RUN_DIR` 의 `scores.jsonl` 과 `thoughts/` 폴더 내용을 저장소에 올리거나(`results/` 폴더), 내용을 붙여넣어 주세요. 분석은 Claude Code가 합니다.